# Image Data Exploration and OCR

Initial work for **DV DS home assignment — CV + NLP**.

The assignment labels images according to alcohol mentions in their text. This notebook covers imports, loading the supplied images, basic statistics, and CPU-only text extraction. All generated tables and figures are saved under `./results/`.

Run from the repository root using the environment described in `README.md`.

## 1. Imports

In [ ]:
from pathlib import Path

import easyocr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from matplotlib.ticker import StrMethodFormatter
from PIL import Image, ImageOps
from tqdm.auto import tqdm

DATA_DIR = Path("images")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

pd.options.display.float_format = "{:,.2f}".format
plt.rcParams.update(
    {"font.size": 16, "axes.titlesize": 20, "axes.labelsize": 18}
)

# Limit CPU threads to avoid excessive overhead on these small images.
torch.set_num_threads(4)

## 2. Load the Data and Explore Basic Statistics

Use the parent folder as the label: `alcohol` or `non_alcohol`. Load each image to check it is readable and collect its dimensions, format, and file size. Keep file paths rather than all image pixels in memory.

### 2.1 Function Definition

In [ ]:
def load_image_dataset(data_dir):
    """Load image paths, folder labels, and basic image metadata.

    Args:
        data_dir (Path): Directory containing alcohol and non_alcohol folders.

    Returns:
        pd.DataFrame: One row per image with its path, label, and metadata.
    """
    records = []
    image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

    for label in ("alcohol", "non_alcohol"):
        for image_path in sorted((data_dir / label).iterdir()):
            if image_path.suffix.lower() not in image_extensions:
                continue
            with Image.open(image_path) as image:
                image.load()
                width, height = ImageOps.exif_transpose(image).size
                records.append(
                    {
                        "file_name": image_path.name,
                        "image_path": str(image_path),
                        "label": label,
                        "width": width,
                        "height": height,
                        "format": image.format,
                        "mode": image.mode,
                        "file_size_kb": image_path.stat().st_size / 1024,
                    }
                )

    return pd.DataFrame(records)

### 2.2 Function Call

In [ ]:
dataset = load_image_dataset(DATA_DIR)

### 2.3 Results and Visualization

In [ ]:
class_summary = dataset.groupby("label").size().rename("image_count").to_frame()
class_summary["percentage"] = class_summary["image_count"] / len(dataset) * 100
image_statistics = dataset[["width", "height", "file_size_kb"]].describe().T

print(f"Total images: {len(dataset):,}")
print(f"Total file size: {dataset['file_size_kb'].sum() / 1024:,.2f} MB")
display(dataset.head())
display(
    class_summary.style.format(
        {"image_count": "{:,.0f}", "percentage": "{:,.2f}"}
    )
)
display(image_statistics)
display(
    dataset.groupby(["format", "mode"]).size().rename("image_count").to_frame()
)

dataset.to_csv(RESULTS_DIR / "image_metadata.csv", index=False)
class_summary.to_csv(RESULTS_DIR / "class_summary.csv")
image_statistics.to_csv(RESULTS_DIR / "image_statistics.csv")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].bar(
    class_summary.index,
    class_summary["image_count"],
    color=["#4878A8", "#59A14F"],
)
axes[0].set(title="Images per Class", xlabel="Class", ylabel="Image count")
axes[0].yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
for position, count in enumerate(class_summary["image_count"]):
    axes[0].text(
        position, count, f"{count:,}", ha="center", va="bottom", fontsize=18
    )
axes[0].set_ylim(0, class_summary["image_count"].max() * 1.15)

axes[1].hist(dataset["width"], bins=15, alpha=0.65, label="Width")
axes[1].hist(dataset["height"], bins=15, alpha=0.65, label="Height")
axes[1].set(title="Image Dimensions", xlabel="Pixels", ylabel="Image count")
axes[1].xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
axes[1].yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
axes[1].legend()
fig.tight_layout()
fig.savefig(
    RESULTS_DIR / "dataset_statistics.png", dpi=150, bbox_inches="tight"
)
plt.show()

# Show the first two images in each class for a reproducible visual check.
examples = dataset.groupby("label", sort=True).head(2)
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for axis, row in zip(axes.flat, examples.itertuples(index=False)):
    with Image.open(row.image_path) as image:
        axis.imshow(ImageOps.exif_transpose(image).convert("RGB"))
    axis.set_title(f"{row.label} | {row.file_name}", fontsize=18)
    axis.axis("off")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "sample_images.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Extract Text with OCR

Use [EasyOCR](https://www.jaided.ai/easyocr/documentation/), one of the libraries suggested in the assignment, with English recognition and `gpu=False`. Initialize the reader once and use its default detection and recognition settings for this first pass. On first use, EasyOCR downloads its pretrained models into `.cache/easyocr/`; later runs reuse them.

Keep recognized text without a confidence cutoff so uncertain words remain available for inspection. Store mean confidence per image on EasyOCR's 0–1 scale; an image without detected text has an empty string and missing confidence. Joining detected text boxes with spaces provides a simple text column, without assuming a reliable reading order.

OCR coverage and confidence describe extraction behavior. They do not measure transcription accuracy because the dataset does not provide reference transcriptions.

### 3.1 Function Definition

In [ ]:
def create_ocr_reader(model_dir):
    """Initialize an English EasyOCR reader for CPU-only execution.

    Args:
        model_dir (Path): Local directory for downloaded OCR model weights.

    Returns:
        easyocr.Reader: Reader shared across all images.
    """
    return easyocr.Reader(
        ["en"],
        gpu=False,
        model_storage_directory=str(model_dir),
        user_network_directory=str(model_dir / "user_network"),
        verbose=False,
    )


def extract_image_text(dataset, reader):
    """Extract text and confidence information from every dataset image.

    Args:
        dataset (pd.DataFrame): Image metadata including image_path.
        reader (easyocr.Reader): Initialized OCR reader.

    Returns:
        pd.DataFrame: Metadata with extracted text, word counts, detected
            text-box counts, and mean recognition confidence per image.
    """
    records = []
    for row in tqdm(
        dataset.itertuples(index=False), total=len(dataset), desc="OCR"
    ):
        with Image.open(row.image_path) as image:
            image_array = np.array(
                ImageOps.exif_transpose(image).convert("RGB")
            )
        detections = reader.readtext(image_array, detail=1, paragraph=False)
        text_parts = [text for _, text, _ in detections if text.strip()]
        confidences = [
            float(confidence)
            for _, text, confidence in detections
            if text.strip()
        ]
        extracted_text = " ".join(text_parts)
        records.append(
            {
                "extracted_text": extracted_text,
                "word_count": len(extracted_text.split()),
                "text_box_count": len(text_parts),
                "mean_confidence": (
                    np.mean(confidences) if confidences else np.nan
                ),
            }
        )

    return pd.concat(
        [dataset.reset_index(drop=True), pd.DataFrame(records)], axis=1
    )

### 3.2 Function Call

In [ ]:
ocr_reader = create_ocr_reader(Path(".cache") / "easyocr")
ocr_results = extract_image_text(dataset, ocr_reader)
ocr_results.to_csv(RESULTS_DIR / "ocr_text.csv", index=False)

### 3.3 Results and Visualization

In [ ]:
ocr_summary = (
    ocr_results.assign(has_text=ocr_results["word_count"].gt(0))
    .groupby("label")
    .agg(
        image_count=("file_name", "size"),
        images_with_text=("has_text", "sum"),
        mean_word_count=("word_count", "mean"),
        mean_confidence=("mean_confidence", "mean"),
    )
)
ocr_summary["images_without_text"] = (
    ocr_summary["image_count"] - ocr_summary["images_with_text"]
)
ocr_summary["text_coverage_percent"] = (
    ocr_summary["images_with_text"] / ocr_summary["image_count"] * 100
)
ocr_summary.to_csv(RESULTS_DIR / "ocr_summary.csv")

print(
    f"Images with extracted text: {ocr_results['word_count'].gt(0).sum():,} / {len(ocr_results):,}"
)
display(ocr_summary)
display(
    ocr_results.groupby("label", sort=True)
    .head(3)[
        [
            "file_name",
            "label",
            "extracted_text",
            "word_count",
            "mean_confidence",
        ]
    ]
    .style.format(
        {"word_count": "{:,.0f}", "mean_confidence": "{:,.2f}"}, na_rep="N/A"
    )
)
print("Images without detected text (up to five examples):")
display(
    ocr_results.loc[
        ocr_results["word_count"].eq(0), ["file_name", "label"]
    ].head()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].bar(
    ocr_summary.index,
    ocr_summary["text_coverage_percent"],
    color=["#4878A8", "#59A14F"],
)
axes[0].set(
    title="OCR Text Coverage",
    xlabel="Class",
    ylabel="Images with text (%)",
    ylim=(0, 115),
)
axes[0].yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
for position, coverage in enumerate(ocr_summary["text_coverage_percent"]):
    axes[0].text(
        position,
        coverage,
        f"{coverage:,.2f}%",
        ha="center",
        va="bottom",
        fontsize=18,
    )

axes[1].hist(ocr_results["word_count"], bins=20, color="#4878A8")
axes[1].set(
    title="Extracted Word Counts",
    xlabel="Words per image",
    ylabel="Image count",
)
axes[1].xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
axes[1].yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
fig.tight_layout()
fig.savefig(RESULTS_DIR / "ocr_statistics.png", dpi=150, bbox_inches="tight")
plt.show()